# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SanniyaAslam/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/flyrank-ml-internship"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/SanniyaAslam/flyrank-ml-internship.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "scikit-learn"], check=True)

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_fh,
               SUM(gsc_clicks) AS clicks_fh,
               AVG(gsc_avg_position) AS avg_position_fh
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND EXTRACT(day FROM report_date) <= 15
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS clicks_sh
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND EXTRACT(day FROM report_date) > 15
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        fh.client_hash_id, fh.content_hash_id,
        fh.impressions_fh, fh.clicks_fh, fh.avg_position_fh,
        sh.clicks_sh,
        d.content_updated_date, d.word_count,
        DATE_DIFF('day', d.content_updated_date, DATE '2026-03-15') AS days_since_update_fh
    FROM first_half fh
    JOIN second_half sh ON fh.client_hash_id = sh.client_hash_id AND fh.content_hash_id = sh.content_hash_id
    JOIN read_parquet('{REL}/dim_content.parquet') d
      ON fh.client_hash_id = d.client_hash_id AND fh.content_hash_id = d.content_hash_id
    WHERE fh.impressions_fh > 0
""").df()

df["ctr_fh"] = df["clicks_fh"] / df["impressions_fh"]
df["is_declining"] = (df["clicks_sh"] < df["clicks_fh"]).astype(int)
df["word_count"] = df["word_count"].fillna(df["word_count"].median())
df["days_since_update_fh"] = df["days_since_update_fh"].clip(lower=0)

features = ["impressions_fh", "ctr_fh", "avg_position_fh", "days_since_update_fh", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining"].values
groups = df["client_hash_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

df_test = df.iloc[test_idx].reset_index(drop=True)
df_test["decline_score"] = rf.predict_proba(X_test)[:, 1]
print("Model trained. Scored rows:", len(df_test))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model trained. Scored rows: 11550


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*The model flagged 4,830 pages as "investigate" (high decline-risk) and 19
as "monitor" (moderate risk), out of 11,550 test pages — zero pages hit
"refresh_now" in this test slice. Looking at the top 10 ranked actions,
every single one carries reason code HIGH_RISK_FRESH, not
HIGH_RISK_STALE: these are pages with very low impressions (15-19 over
the half-month) and low-to-moderate CTR, most with days_since_update_fh
= 0 (i.e. recently touched content). This is a genuinely useful finding
for a reviewer: the highest-risk pages in this slice are not simply
"old, forgotten content" — they're low-visibility pages that may never
have gained real traction, regardless of freshness. A reviewer should
read "investigate" as "check why this page isn't getting traction" rather
than assuming a refresh alone will fix it, since staleness isn't the
driver here.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def reason_and_action(row):
    if row["decline_score"] >= 0.7 and row["days_since_update_fh"] >= 180:
        return "HIGH_RISK_STALE", "refresh_now"
    elif row["decline_score"] >= 0.7:
        return "HIGH_RISK_FRESH", "investigate"
    elif row["decline_score"] >= 0.4:
        return "MODERATE_RISK", "monitor"
    else:
        return "LOW_RISK", "no_action"

df_test[["reason_code", "action"]] = df_test.apply(
    lambda r: pd.Series(reason_and_action(r)), axis=1
)

playbook_queue = df_test.sort_values("decline_score", ascending=False).reset_index(drop=True)

print("Action distribution:")
print(playbook_queue["action"].value_counts())
print("\n=== Top 10 ranked actions ===")
display(playbook_queue.head(10)[["content_hash_id", "impressions_fh", "ctr_fh",
                                    "avg_position_fh", "days_since_update_fh",
                                    "decline_score", "reason_code", "action"]])

Action distribution:
action
no_action      6701
investigate    4830
monitor          19
Name: count, dtype: int64

=== Top 10 ranked actions ===


,content_hash_id,impressions_fh,ctr_fh,avg_position_fh,days_since_update_fh,decline_score,reason_code,action
0,content_a3379fade728f73d,15.0,0.066667,8.692857,0,0.920333,HIGH_RISK_FRESH,investigate
1,content_fb5b22f6eff0507f,15.0,0.066667,23.366667,18,0.920199,HIGH_RISK_FRESH,investigate
2,content_2a0482a861d57d5d,1.0,1.000000,9.000000,0,0.919798,HIGH_RISK_FRESH,investigate
3,content_3e328be0aa8b5e52,16.0,0.062500,20.888889,0,0.919318,HIGH_RISK_FRESH,investigate
4,content_ca84f1c0a5a3786b,16.0,0.062500,13.796296,0,0.918886,HIGH_RISK_FRESH,investigate
5,content_f3c58a5bf1af50e4,17.0,0.117647,8.166667,0,0.918837,HIGH_RISK_FRESH,investigate
6,content_2cd25898f7cf2f5c,16.0,0.062500,8.750000,0,0.918101,HIGH_RISK_FRESH,investigate
7,content_121e3f7e8a310107,19.0,0.052632,7.638889,18,0.914828,HIGH_RISK_FRESH,investigate
8,content_21d3f3eba36877e4,18.0,0.055556,7.353333,0,0.914723,HIGH_RISK_FRESH,investigate
9,content_824ae6398a850550,15.0,0.066667,14.000000,0,0.914367,HIGH_RISK_FRESH,investigate


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*Intended use: this playbook is decision-support for a content strategist
or SEO reviewer managing a portfolio of pages, helping them prioritize a
limited weekly review capacity. It is NOT an automated content-editing
tool and does not publish, delete, or modify anything on its own.

Limits: the model was trained on one client-grouped slice of March 2026
data, using a within-month (first-half vs second-half) proxy label —
performance may not generalize to other months, seasons, or a longer-term
decline pattern. It also cannot distinguish "content that needs a
refresh" from "content facing a temporary external factor" (a
competitor's new page, a SERP feature change, a seasonal dip) — those
require human judgment the model has no signal for. The playbook is only
as current as its March 2026 training window; it stops being valid once
site content, competitive landscape, or algorithm behavior shifts
meaningfully.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended use and limits documented above — no additional query needed.")

Intended use and limits documented above — no additional query needed.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*Before acting on any "refresh_now" flag, a human must check: (1) whether
the page's traffic drop coincides with a known external event (algorithm
update, competitor launch, seasonality) rather than genuine content
decay; (2) whether the page still matches current business priorities —
a page can be "declining" and also no longer strategically relevant; (3)
sample size — pages with very low impressions (under ~50-100) should be
treated with lower confidence, since CTR estimates on small samples are
noisy (observed directly in my Week-5/6 error analysis, where several
false alarms had single-digit impression counts).

What should NEVER be automated: (1) publishing or unpublishing content
based on the score alone; (2) editing page content without a human
writer/editor reviewing the specific page; (3) treating the decline_score
as a ranking-algorithm prediction — it estimates observed traffic
decline risk, not Google's actual ranking logic; (4) using this queue to
make client-facing claims about "guaranteed" recovery from a refresh —
the underlying evidence (my Week-5/6 model, and FlyRank's own paper
Finding #4) is observational, not causal.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Human review rules and no-go list documented above — no additional query needed.")

Human review rules and no-go list documented above — no additional query needed.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*Retrain the model if: (1) the declining-page rate drifts meaningfully
from the ~20% baseline observed in March 2026 (a sign the label
distribution has shifted); (2) Precision@50 on a fresh month's holdout
drops notably below the 0.74-0.88 range seen in Week 5/6 testing; (3) a
new client onboards with a very different traffic profile than the 104
clients in the current warehouse, since the model has never seen that
pattern; (4) three consecutive months pass without revalidation — set as
a simple time-based trigger since content and search behavior drift
gradually even without an obvious break.

Monitoring signal to watch weekly: the count of pages flagged
"refresh_now" — a sudden spike or drop suggests either a real portfolio-
wide shift or a data-quality issue (e.g. the days_since_update
negative-value quirk found in Week 3/4) rather than a real change in
content health.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Monitoring and retrain triggers documented above — no additional query needed.")


Monitoring and retrain triggers documented above — no additional query needed.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Full ranked queue (gitignored, regenerated each run)
playbook_queue[["client_hash_id", "content_hash_id", "impressions_fh", "ctr_fh",
                "avg_position_fh", "days_since_update_fh", "decline_score",
                "reason_code", "action"]].to_csv("work/outputs/action_playbook_queue.csv", index=False)

# Metrics JSON — the "receipts" that get committed
import json
metrics = {
    "model": "RandomForestClassifier (n_estimators=200, max_depth=6)",
    "split": "client-grouped (GroupShuffleSplit, test_size=0.25)",
    "test_rows": len(df_test),
    "action_counts": playbook_queue["action"].value_counts().to_dict(),
    "declining_rate_in_test": float(y_test.mean()),
}
with open("work/outputs/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported:")
print("- work/outputs/action_playbook_queue.csv (gitignored, regenerated)")
print("- work/outputs/w07_playbook_metrics.json (commit this)")
print(json.dumps(metrics, indent=2))

Exported:
- work/outputs/action_playbook_queue.csv (gitignored, regenerated)
- work/outputs/w07_playbook_metrics.json (commit this)
{
  "model": "RandomForestClassifier (n_estimators=200, max_depth=6)",
  "split": "client-grouped (GroupShuffleSplit, test_size=0.25)",
  "test_rows": 11550,
  "action_counts": {
    "no_action": 6701,
    "investigate": 4830,
    "monitor": 19
  },
  "declining_rate_in_test": 0.25203463203463206
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.